# FP4 quantization-aware training in DeepSeek-V4
### Notebook 7 of 9 on DeepSeek-V4 (technical report [arXiv:2606.19348](https://arxiv.org/abs/2606.19348), §5.2.1)

Serving a 1.6-trillion-parameter model is mostly a memory problem: the expert weights are most of the parameters (notebook 4: about 97 % of the total). V4 therefore introduces **quantization-aware training (QAT)** in the post-training stage, so that the model, "including those of teacher and reference models", adapts to 4-bit precision before it is deployed. It applies FP4 in the **Microscaling (MX) format MXFP4** ([Rouhani et al., 2023](https://arxiv.org/abs/2310.10537)) to two places: the **MoE expert weights**, and the **query-key path of the lightning indexer in CSA** (notebook 1), where QK activations are "cached, loaded, and multiplied entirely in FP4". The index scores are additionally cast from FP32 to BF16, which "achieves a 2× speedup for the top-k selector, while preserving a 99.7 % recall rate of KV entries".

The report's recipe for the weights, in its own words: the FP32 master weights "are first quantized to FP4, then dequantized back to FP8 for computation"; this FP4→FP8 dequantization "is lossless" provided the ratio between the largest and smallest scale factors of the FP4 sub-blocks (1×32 tiles) inside each FP8 quantization block (128×128 tiles) does not exceed a threshold, which "we empirically verify that current weights satisfy"; the backward pass uses the same FP8 weights and passes gradients straight to the FP32 masters (a straight-through estimator, STE). During RL rollouts, native FP4 weights are used, so "model behavior during sampling is fully consistent with online deployment".

**What we do.** Build the format from its definition, **test the "lossless" claim against PyTorch's real FP8 (E4M3) type and find the threshold**, implement the STE, and compare QAT with post-training quantization on a small problem. We did not quantize a language model: our results say nothing about V4's accuracy after QAT.

| section | question | number it carries |
|---|---|---|
| 1 | What is an FP4 number, and what is a block scale? | 15 representable values; 4.25 bits/weight |
| 2 | How much error does MXFP4 introduce? | signal-to-quantization-noise ratio |
| 3 | When is FP4 → FP8 exactly lossless? | the scale-ratio threshold |
| 4 | How does the straight-through estimator work? | gradient identity |
| 5 | Does QAT beat post-training quantization? | loss after quantization |
| 6 | What does BF16 do to top-k selection? | recall |
| 7 | What does 4 bits buy at V4's size? | GB for the experts |

In [1]:
import math, time, itertools
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
plt.rcParams.update({"figure.dpi": 105, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25, "font.size": 9})
CHECKS = []
def check(name, ok, detail=""): CHECKS.append((name, bool(ok), detail)); print(("PASS " if ok else "FAIL ") + name + (f"   [{detail}]" if detail else ""))

## An FP4 number and a block scale

**E2M1** is a 4-bit float: 1 sign bit, 2 exponent bits, 1 mantissa bit. Enumerating all patterns gives the magnitudes `{0, 0.5, 1, 1.5, 2, 3, 4, 6}` and their negatives: 15 distinct values (`+0` and `−0` coincide in value). That is a tiny, non-uniform grid: fine near zero, coarse near 6.

A weight tensor does not live on that grid, so MX formats add a **shared scale per block of 32 consecutive elements**, stored as an 8-bit power of two (E8M0). The stored value is `scale × grid_value`, so each block can use the whole grid however large or small its weights are. Tiny example: block maximum 5.1; the OCP MX rule picks the scale `2^(floor(log2(5.1)) − 2) = 2^0 = 1`, so values are rounded onto the grid directly: `5.1 → 4` or `6`? The nearest grid value to 5.1 is `6` (distance 0.9, against 1.1 to `4`), so `5.1 → 6`. If the block maximum were 0.51 the scale would be `2^(−1−2) = 0.125` and `0.51/0.125 = 4.08 → 4`, giving `0.5`.

Storage: 4 bits per element plus 8 bits per 32 elements is `4 + 8/32 = 4.25` bits per weight, against 16 for BF16: 3.76× smaller.

In [2]:
FP4_POS = torch.tensor([0., 0.5, 1., 1.5, 2., 3., 4., 6.])                           # E2M1 magnitudes
def fp4_levels():
    vals = set()
    for s in (1, -1):
        for e in range(4):
            for m in range(2):
                v = (m * 0.5 if e == 0 else (1 + 0.5 * m) * 2 ** (e - 1))              # exponent bias 1: subnormal when e == 0
                vals.add(s * v)
    return sorted(vals)
lv = fp4_levels(); print("E2M1 values:", lv)
check("E2M1 has 15 distinct values, maximum magnitude 6, and its positive magnitudes are {0,.5,1,1.5,2,3,4,6}", len(lv) == 15 and max(lv) == 6 and sorted(set(abs(v) for v in lv)) == FP4_POS.tolist())

def round_to_fp4(x):                                                                  # nearest grid value, preserving sign (x already divided by the block scale)
    mag = x.abs().clamp(max=6.0); idx = (mag[..., None] - FP4_POS).abs().argmin(-1); return torch.sign(x) * FP4_POS[idx]

def mxfp4_quantize(x, block=32):
    """x (..., K) with K % block == 0 -> (dequantised values, scales).  scale = 2^(floor(log2 amax) - 2)  (E2M1 max exponent is 2)."""
    shp = x.shape; xb = x.reshape(*shp[:-1], shp[-1] // block, block)
    amax = xb.abs().amax(-1, keepdim=True).clamp(min=1e-30); scale = 2.0 ** (torch.floor(torch.log2(amax)) - 2)
    q = round_to_fp4(xb / scale) * scale; return q.reshape(shp), scale.squeeze(-1)
b1 = torch.tensor([[5.1, -2.4, 0.3, 1.0] + [0.0] * 28]); q1, s1 = mxfp4_quantize(b1); print("block max 5.1 -> scale", s1.item(), " 5.1 ->", q1[0, 0].item(), " -2.4 ->", q1[0, 1].item(), " 0.3 ->", q1[0, 2].item())
check("worked example: a block with maximum 5.1 gets scale 1 and 5.1 rounds to 6", s1.item() == 1.0 and q1[0, 0].item() == 6.0)
b2 = b1 * 0.1; q2, s2 = mxfp4_quantize(b2); check("a block with maximum 0.51 gets scale 0.125 and 0.51 rounds to 0.5", abs(s2.item() - 0.125) < 1e-9 and abs(q2[0, 0].item() - 0.5) < 1e-9, f"scale {s2.item()}, value {q2[0, 0].item()}")
torch.manual_seed(0); W = torch.randn(64, 256); Wq, _ = mxfp4_quantize(W)
check("every dequantised value is (grid value) x (power-of-two block scale)", bool(torch.isin((Wq.reshape(64, 8, 32) / mxfp4_quantize(W)[1][..., None]).abs().round(decimals=5), FP4_POS).all()))
print(f"bits per weight: {4 + 8 / 32}  ->  {16 / (4 + 8 / 32):.2f}x smaller than BF16")

E2M1 values: [-6.0, -4.0, -3.0, -2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0]
PASS E2M1 has 15 distinct values, maximum magnitude 6, and its positive magnitudes are {0,.5,1,1.5,2,3,4,6}
block max 5.1 -> scale 1.0  5.1 -> 6.0  -2.4 -> -2.0  0.3 -> 0.5
PASS worked example: a block with maximum 5.1 gets scale 1 and 5.1 rounds to 6
PASS a block with maximum 0.51 gets scale 0.125 and 0.51 rounds to 0.5   [scale 0.125, value 0.5]
PASS every dequantised value is (grid value) x (power-of-two block scale)
bits per weight: 4.25  ->  3.76x smaller than BF16


## How much error does it add?

The standard figure is the **signal-to-quantization-noise ratio** `SQNR = 10·log10( E[w²] / E[(w − q(w))²] )` in dB (higher is better). We compare MXFP4 with the most naive 4-bit alternative, symmetric INT4 with a **single scale for the whole tensor**, and with INT4 using the same 32-element blocks, on two weight distributions: Gaussian, and heavy-tailed (Student-t with 3 degrees of freedom) which has outliers.

In [3]:
def int4_quant(x, block=None):
    if block is None: s = x.abs().max() / 7; return (x / s).round().clamp(-8, 7) * s
    shp = x.shape; xb = x.reshape(*shp[:-1], shp[-1] // block, block); s = xb.abs().amax(-1, keepdim=True) / 7
    return ((xb / s).round().clamp(-8, 7) * s).reshape(shp)
sqnr = lambda w, q: 10 * math.log10((w ** 2).mean().item() / ((w - q) ** 2).mean().item())
torch.manual_seed(0); dists = {"Gaussian": torch.randn(256, 1024), "Student-t (3 dof, outliers)": torch.distributions.StudentT(3.0).sample((256, 1024))}
print(f"{'weights':30s} {'INT4 per-tensor':>16s} {'INT4 block-32':>14s} {'MXFP4 block-32':>15s}")
R = {}
for nm, w in dists.items():
    R[nm] = (sqnr(w, int4_quant(w)), sqnr(w, int4_quant(w, 32)), sqnr(w, mxfp4_quantize(w)[0])); print(f"{nm:30s} {R[nm][0]:14.1f}dB {R[nm][1]:12.1f}dB {R[nm][2]:13.1f}dB")
check("block scaling matters: on heavy-tailed weights MXFP4 beats per-tensor INT4 by >5 dB", R["Student-t (3 dof, outliers)"][2] > R["Student-t (3 dof, outliers)"][0] + 5, f"{R['Student-t (3 dof, outliers)'][2]:.1f} vs {R['Student-t (3 dof, outliers)'][0]:.1f} dB")
check("on Gaussian weights the 4-bit options are all within ~10 dB of each other and well above 10 dB", all(10 < v < 30 for v in R["Gaussian"]))

weights                         INT4 per-tensor  INT4 block-32  MXFP4 block-32
Gaussian                                 14.3dB         20.2dB          18.8dB
Student-t (3 dof, outliers)               1.6dB         16.7dB          16.9dB
PASS block scaling matters: on heavy-tailed weights MXFP4 beats per-tensor INT4 by >5 dB   [16.9 vs 1.6 dB]
PASS on Gaussian weights the 4-bit options are all within ~10 dB of each other and well above 10 dB


**What we found.** Per-tensor INT4 is crushed by outliers (one large weight sets the scale for the whole tensor), and 32-element blocks recover most of the loss, which is the point of the MX family. Between INT4 and E2M1 *with the same blocks* the difference is smaller and depends on the distribution; we claim only what the table shows. The question that matters for the V4 recipe is the next one.

## When is FP4 → FP8 exactly lossless?

V4's QAT keeps the existing FP8 training stack. The weight is stored as FP4 values with a scale per 1×32 sub-block, but the matrix multiply runs in FP8, so the FP4 representation has to be converted to FP8 **without changing any value**. E4M3 (4 exponent bits, 3 mantissa bits, maximum 448) has two more exponent bits than E2M1, and every E2M1 magnitude (`0.5, 1, 1.5, 2, 3, 4, 6`) has at most one mantissa bit, so each fits in E4M3's mantissa; what can go wrong is the **exponent range**. In a 128×128 FP8 block there is one FP8 scale `s_blk`; each 1×32 sub-block has its own FP4 scale `s_sub`, and an element's FP8 value is `fp4 × (s_sub / s_blk)`. If `s_sub / s_blk` is very small the product underflows E4M3's smallest subnormal (`2^−9`); if very large it overflows 448. For power-of-two scales the representable window is a fixed number of binades, which is the "certain threshold" on the ratio of the largest to the smallest sub-block scale in the block.

We test it with PyTorch's `float8_e4m3fn` dtype (round-trip through it must reproduce the FP4 values exactly) and sweep the ratio.

In [4]:
def fp8_roundtrip(x): return x.to(torch.float8_e4m3fn).to(torch.float32)
def lossless_for_ratio(ratio_log2, trials=200, seed=0):
    """Build a 128x128 block of FP4 values with 1x32 sub-block scales spanning 2^ratio_log2 between the largest and smallest; return the fraction of blocks whose FP8 image is exact."""
    g = torch.Generator().manual_seed(seed); ok = 0
    for _ in range(trials):
        fp4 = FP4_POS[torch.randint(0, 8, (128, 128), generator=g)] * (torch.randint(0, 2, (128, 128), generator=g) * 2 - 1)
        e = torch.randint(0, ratio_log2 + 1, (128, 4), generator=g); e[0, 0] = 0; e[0, 1] = ratio_log2                                         # sub-block exponents span exactly ratio_log2 binades
        s_sub = (2.0 ** (-e.float())).repeat_interleave(32, dim=1)                                                                             # the largest sub-block scale is 1 (= s_blk); others smaller
        val = fp4 * s_sub                                                                                                                     # what the FP8 block must hold, relative to s_blk
        ok += int(torch.equal(fp8_roundtrip(val), val))
    return ok / trials
print(f"{'max/min sub-block scale':>24s} {'fraction of blocks exactly representable in FP8 E4M3':>56s}")
LL = {r: lossless_for_ratio(r) for r in [0, 2, 4, 6, 8, 9, 10, 12, 16]}
for r, f in LL.items(): print(f"{'2^' + str(r) + ' = ' + str(2 ** r):>24s} {f:56.2f}")
thr = max(r for r, f in LL.items() if f == 1.0); print(f"\nlargest ratio in this sweep that is still lossless in every trial: 2^{thr} = {2 ** thr}")
check("small scale ratios (up to 2^4) are exactly representable in FP8 E4M3", all(LL[r] == 1.0 for r in (0, 2, 4)))
check("a threshold exists: very large ratios (2^12 and above) are never lossless", LL[12] == 0.0 and LL[16] == 0.0)
check("our measured threshold sits at 2^8 (E4M3: smallest subnormal 2^-9 vs the smallest FP4 magnitude 0.5 = 2^-1)", thr == 8, f"2^{thr}")

 max/min sub-block scale     fraction of blocks exactly representable in FP8 E4M3


                 2^0 = 1                                                     1.00
                 2^2 = 4                                                     1.00
                2^4 = 16                                                     1.00
                2^6 = 64                                                     1.00
               2^8 = 256                                                     1.00
               2^9 = 512                                                     0.00
             2^10 = 1024                                                     0.00
             2^12 = 4096                                                     0.00
            2^16 = 65536                                                     0.00

largest ratio in this sweep that is still lossless in every trial: 2^8 = 256
PASS small scale ratios (up to 2^4) are exactly representable in FP8 E4M3
PASS a threshold exists: very large ratios (2^12 and above) are never lossless
PASS our measured threshold sit

**What we found.** The claim is true and has a concrete threshold: **for power-of-two scales, the FP4 values inside a 128×128 block are exactly representable in E4M3 as long as the largest sub-block scale is at most `2^8 = 256` times the smallest** in that block; at `2^9` no block in our trials was lossless, and the same holds at every larger ratio we tried: the transition is sharp. The reasoning is the exponent arithmetic above: the smallest nonzero FP4 magnitude (`0.5 = 2^−1`) times the smallest ratio must stay at or above E4M3's smallest subnormal `2^−9`, leaving 8 binades. The report states that "current weights satisfy this condition" without giving the threshold or the observed ratios; we do not have V4's weights, so we cannot say how much margin real weights have. (We assumed the block's FP8 scale is the largest sub-block scale, our choice of reference; the report does not specify it.) What this buys is the report's point: QAT can reuse the FP8 training framework unchanged.

## The straight-through estimator

Rounding to a grid has zero gradient almost everywhere, so back-propagating through `q(w)` would stop learning. The straight-through estimator treats the quantizer as the identity in the backward pass: `∂L/∂w := ∂L/∂q(w)`. The forward pass uses the quantized weight `q(w)` (so the loss sees the quantization error and the network can adapt to it); the optimizer updates the FP32 master `w`. The report states the equivalent: "gradients are computed with respect to the same FP8 weights in the forward pass and directly propagated back to the FP32 master weights". A convenient fact: because the gradient is taken at the quantized weight, no re-quantization of transposed weights is needed in the backward pass.

In [5]:
class MXFP4STE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, w): return mxfp4_quantize(w)[0]                           # forward: the quantised weight
    @staticmethod
    def backward(ctx, g): return g                                             # backward: straight through to the FP32 master
class QLinear(nn.Module):
    def __init__(s, i, o, quant=True): super().__init__(); s.w = nn.Parameter(torch.randn(o, i) * i ** -.5); s.b = nn.Parameter(torch.zeros(o)); s.quant = quant
    def forward(s, x): return F.linear(x, MXFP4STE.apply(s.w) if s.quant else s.w, s.b)

torch.manual_seed(0); ql = QLinear(64, 8); x = torch.randn(5, 64); ql(x).pow(2).sum().backward(); g_ste = ql.w.grad.clone()
wq = mxfp4_quantize(ql.w.detach())[0].requires_grad_(True); F.linear(x, wq, ql.b.detach()).pow(2).sum().backward()
check("STE: the gradient reaching the FP32 master equals the gradient w.r.t. the quantised weight", torch.allclose(g_ste, wq.grad, atol=1e-6))
w_naive = ql.w.detach().clone().requires_grad_(True); (round_to_fp4(w_naive / 1.0) * x[0, :64].unsqueeze(0).expand(8, 64)).sum().backward()
check("without the STE, back-propagating through the rounding gives a zero gradient (so QAT could not learn)", w_naive.grad.abs().max().item() == 0.0, f"max |grad| = {w_naive.grad.abs().max().item()}")

PASS STE: the gradient reaching the FP32 master equals the gradient w.r.t. the quantised weight
PASS without the STE, back-propagating through the rounding gives a zero gradient (so QAT could not learn)   [max |grad| = 0.0]


## QAT against post-training quantization

Two ways to get a 4-bit model. **Post-training quantization (PTQ)**: train in full precision, then round the weights once. **QAT**: train with the quantizer in the loop (forward uses `q(w)`, STE backward), so the weights settle where rounding hurts least. We compare on a small regression problem: a 3-layer MLP student (64 → 256 → 256 → 16, `tanh`) learning a random teacher. All three variants start from the same initialization and see the same data; the fully-trained FP32 weights are the starting point for PTQ's rounding. Every weight matrix is quantized to MXFP4 along its input dimension in blocks of 32 (biases stay FP32).

In [6]:
def make(quant):
    torch.manual_seed(0); return nn.Sequential(QLinear(64, 256, quant), nn.Tanh(), QLinear(256, 256, quant), nn.Tanh(), QLinear(256, 16, quant))
torch.manual_seed(1); Tch = nn.Sequential(nn.Linear(64, 256), nn.Tanh(), nn.Linear(256, 256), nn.Tanh(), nn.Linear(256, 16)); X = torch.randn(8192, 64); Y = Tch(X).detach(); Xt = torch.randn(2048, 64); Yt = Tch(Xt).detach()
def fit(model, steps=1200, lr=3e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr); torch.manual_seed(2)
    for s in range(steps):
        idx = torch.randint(0, 8192, (256,)); loss = F.mse_loss(model(X[idx]), Y[idx]); opt.zero_grad(); loss.backward(); opt.step()
    return model
def test_mse(model):
    with torch.no_grad(): return F.mse_loss(model(Xt), Yt).item()
fp32 = fit(make(False)); mse_fp32 = test_mse(fp32)
ptq = make(True); ptq.load_state_dict(fp32.state_dict()); mse_ptq = test_mse(ptq)                  # same FP32 weights, but the forward now rounds them to MXFP4
qat = fit(make(True)); mse_qat = test_mse(qat)
var = Yt.var().item()
print(f"{'model':28s} {'held-out MSE':>14s} {'relative to target variance':>28s}")
for nm, v in [("FP32", mse_fp32), ("PTQ (round FP32 weights to MXFP4)", mse_ptq), ("QAT (train with MXFP4 + STE)", mse_qat)]: print(f"{nm:28s} {v:14.5f} {v / var:28.4f}")
check("post-training rounding to MXFP4 adds clear error on this problem (PTQ MSE > 1.3x the FP32 MSE)", mse_ptq > 1.3 * mse_fp32, f"{mse_ptq / mse_fp32:.1f}x")
check("QAT recovers most of that: its MSE is below PTQ's and within 2x of FP32", mse_qat < mse_ptq and mse_qat < 2 * mse_fp32, f"{mse_qat:.5f} vs {mse_ptq:.5f}")
print(f"QAT closes {100 * (mse_ptq - mse_qat) / (mse_ptq - mse_fp32):.0f}% of the gap between PTQ and FP32;  it still sits {mse_qat / mse_fp32:.1f}x above FP32.")

model                          held-out MSE  relative to target variance
FP32                                0.00080                       0.0349
PTQ (round FP32 weights to MXFP4)        0.00125                       0.0543
QAT (train with MXFP4 + STE)        0.00084                       0.0366
PASS post-training rounding to MXFP4 adds clear error on this problem (PTQ MSE > 1.3x the FP32 MSE)   [1.6x]
PASS QAT recovers most of that: its MSE is below PTQ's and within 2x of FP32   [0.00084 vs 0.00125]
QAT closes 91% of the gap between PTQ and FP32;  it still sits 1.0x above FP32.


**What we found.** Rounding a trained FP32 model to MXFP4 costs accuracy on this small problem, and training with the quantizer in the loop recovers about 90 % of that gap, ending within roughly 5 % of the FP32 error (numbers above). That is the case for doing QAT rather than quantizing at the end; V4's authors add a reason specific to their pipeline, that teachers and reference models are also quantized so that the whole post-training loop sees consistent numerics. This is one seed on a smooth regression task; it does not tell us how much accuracy V4 loses or recovers, and the report does not give a before-and-after comparison in the sections we read.

## BF16 index scores and top-k recall

The lightning indexer produces one score per candidate compressed block, and the top-`k` are selected (notebook 1). The report casts those scores from FP32 to BF16 and reports that the selector runs 2× faster while keeping **99.7 % recall** of the KV entries that FP32 would have selected. BF16 keeps 8 bits of significand precision (relative step `2^−8 ≈ 0.4 %`), so the only selections that can change are those near the cut-off, where two scores are within a BF16 rounding step of each other. We measure the recall for random scores, where the density of near-ties is set by the score distribution, so this is a *mechanism* check and not a reproduction of the 99.7 % (that depends on the real indexer's score distribution, which we do not have).

In [7]:
def topk_recall(n, k, scale, seed=0):
    g = torch.Generator().manual_seed(seed); s = torch.randn(n, generator=g) * scale; ref = set(s.topk(k).indices.tolist()); bf = set(s.bfloat16().float().topk(k).indices.tolist()); return len(ref & bf) / k
print(f"{'n candidates':>13s} {'k':>6s} {'score std':>10s} {'recall of BF16 top-k vs FP32':>30s}")
rec = {}
for n_, k_ in [(16384, 512), (65536, 1024), (250000, 1024)]:
    for sc in [1.0, 10.0]:
        rec[(n_, sc)] = np.mean([topk_recall(n_, k_, sc, seed) for seed in range(5)]); print(f"{n_:13d} {k_:6d} {sc:10.1f} {rec[(n_, sc)]:30.4f}")
check("BF16 index scores keep > 98% of the FP32 top-k on random scores at V4-like sizes", all(v > 0.98 for v in rec.values()), f"min {min(rec.values()):.4f}")
check("recall does not reach 100% (rounding does flip near-ties at the cut-off)", any(v < 1.0 for v in rec.values()))

 n candidates      k  score std   recall of BF16 top-k vs FP32
        16384    512        1.0                         0.9973
        16384    512       10.0                         0.9953
        65536   1024        1.0                         0.9951
        65536   1024       10.0                         0.9971
       250000   1024        1.0                         0.9924
       250000   1024       10.0                         0.9967
PASS BF16 index scores keep > 98% of the FP32 top-k on random scores at V4-like sizes   [min 0.9924]
PASS recall does not reach 100% (rounding does flip near-ties at the cut-off)


**What we found.** On random scores BF16 keeps well above 98 % of the FP32 selection and is not lossless, in the same range as the report's 99.7 % on its real indexer; the dropped entries are the ones whose scores were within a rounding step of the cut-off, which are by construction nearly as good as the ones kept. The 2× speed-up is a hardware effect we cannot measure on a CPU.

## What 4 bits buys at V4's size

In [8]:
bits = 4 + 8 / 32
for nm, params in [("V4-Flash experts", 278.1e9), ("V4-Pro experts", 1551.4e9)]:
    print(f"{nm:18s}: BF16 {params * 2 / 1e9:8.0f} GB   FP8 {params * 1 / 1e9:8.0f} GB   MXFP4 {params * bits / 8 / 1e9:8.0f} GB   ({16 / bits:.2f}x smaller than BF16, {8 / bits:.2f}x than FP8)")
check("MXFP4 at 4.25 bits/weight is 3.76x smaller than BF16 and 1.88x smaller than FP8", abs(16 / bits - 3.7647) < 1e-3 and abs(8 / bits - 1.8824) < 1e-3)

V4-Flash experts  : BF16      556 GB   FP8      278 GB   MXFP4      148 GB   (3.76x smaller than BF16, 1.88x than FP8)
V4-Pro experts    : BF16     3103 GB   FP8     1551 GB   MXFP4      824 GB   (3.76x smaller than BF16, 1.88x than FP8)
PASS MXFP4 at 4.25 bits/weight is 3.76x smaller than BF16 and 1.88x smaller than FP8


The expert-parameter counts come from our own arithmetic in notebook 4 (`278B` and `1.55T`); the report does not state a quantized model size in the sections we read.

## Closing thoughts

The two numerical claims in the report's QAT recipe check out in our tests: FP4→FP8 conversion is exactly lossless below a concrete scale-ratio threshold (we measured `2^8` for power-of-two scales, under our assumed reference scale), and a straight-through estimator lets the FP32 master weights learn through the quantizer. QAT beat post-training rounding on our small problem (closing about 91 % of the gap to FP32), and BF16 index scores kept 99.2-99.7 % of the top-k selection on random scores, close to the report's 99.7 % on its real indexer.

**What is open.** The report states that its weights satisfy the scale-ratio condition but gives neither the threshold nor the observed ratios, and gives no accuracy before and after QAT in the sections we read. We did not quantize an LLM, did not test FP4 for the indexer's query-key path (only the score precision), and could not measure the speed-ups, which are properties of the hardware kernels.

**Next.** Notebook 8 turns to the other half of the systems story: how the MoE layer overlaps communication with computation.

Credits: MXFP4 is the OCP Microscaling format (Rouhani et al., 2023); the V4 QAT recipe and numbers are DeepSeek-AI's.

In [9]:
print(f"{'#':>2s}  {'status':6s}  check")
for i, (n_, ok, d_) in enumerate(CHECKS, 1): print(f"{i:2d}  {'PASS' if ok else 'FAIL':6s}  {n_}")
failed = [n_ for n_, ok, _ in CHECKS if not ok]; print(f"\n{len(CHECKS) - len(failed)} passed, {len(failed)} failed")
assert not failed, failed

 #  status  check
 1  PASS    E2M1 has 15 distinct values, maximum magnitude 6, and its positive magnitudes are {0,.5,1,1.5,2,3,4,6}
 2  PASS    worked example: a block with maximum 5.1 gets scale 1 and 5.1 rounds to 6
 3  PASS    a block with maximum 0.51 gets scale 0.125 and 0.51 rounds to 0.5
 4  PASS    every dequantised value is (grid value) x (power-of-two block scale)
 5  PASS    block scaling matters: on heavy-tailed weights MXFP4 beats per-tensor INT4 by >5 dB
 6  PASS    on Gaussian weights the 4-bit options are all within ~10 dB of each other and well above 10 dB
 7  PASS    small scale ratios (up to 2^4) are exactly representable in FP8 E4M3
 8  PASS    a threshold exists: very large ratios (2^12 and above) are never lossless
 9  PASS    our measured threshold sits at 2^8 (E4M3: smallest subnormal 2^-9 vs the smallest FP4 magnitude 0.5 = 2^-1)
10  PASS    STE: the gradient reaching the FP32 master equals the gradient w.r.t. the quantised weight
11  PASS    without the STE, 